# 🔍 Análisis de Interpretabilidad con SHAP
**Credit Card Churn Prediction** — Notebook 03

---

## ¿Por qué necesitamos interpretabilidad?

En el Notebook 02 entrenamos un **Random Forest** con resultados excelentes:

| Métrica | Valor |
|---------|-------|
| AUC-ROC | **0.985** |
| Accuracy | **96%** |
| Recall (churn) | **81%** |

Sin embargo, en un entorno bancario real, predecir bien no es suficiente.
El equipo de retención necesita saber **por qué** el modelo marca a un cliente como en riesgo
para poder tomar acciones concretas.

**SHAP (SHapley Additive exPlanations)** nos permite:
- 🌍 **Globalmente**: qué variables tienen mayor impacto en todas las predicciones
- 🎯 **Individualmente**: por qué el modelo clasifica a *este* cliente como churner
- 📈 **En detalle**: cómo varía el impacto de una variable según su valor

> SHAP proviene de la teoría de juegos: distribuye "crédito" entre variables de la misma
> manera que se reparten las ganancias en un juego cooperativo.

---
## 1. Preparación: Datos y Modelo

Antes de analizar el modelo con SHAP necesitamos:
1. Cargar y preprocesar los datos (mismo proceso que en el Notebook 02)
2. Reentrenar el Random Forest

> 💡 **Nota de ejecución**: asegúrate de correr este notebook desde la raíz del proyecto
> (la carpeta donde vive `data_limpia.csv`).

In [ ]:
# Si SHAP no está instalado, descomenta la siguiente línea y ejecuta:
# !pip install shap

# ── Librerías estándar ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Machine Learning ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ── Interpretabilidad ─────────────────────────────────────────────────────
import shap

# ── Configuración de gráficos ─────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías cargadas correctamente')


In [ ]:
# ── Cargar datos ──────────────────────────────────────────────────────────
# Ruta relativa: ejecuta el notebook desde la raíz del proyecto
df = pd.read_csv("data_limpia.csv")

# ── Eliminar columna auxiliar creada en el EDA ───────────────────────────
df = df.drop(columns=['Age_Group'])

# ── Encoding de variables categóricas ────────────────────────────────────
# Las variables de texto se convierten a números para que el modelo las entienda.
# Usamos LabelEncoder: asigna un número entero a cada categoría.
encoder = LabelEncoder()
categorical_cols = ['Gender', 'Education_Level', 'Marital_Status',
                    'Income_Category', 'Card_Category']

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

# ── Separar features (X) y target (y) ────────────────────────────────────
X = df.drop(columns=['Churn'])   # Variables predictoras (19 variables)
y = df['Churn']                  # Variable objetivo: 0 = retenido, 1 = churned

# ── División entrenamiento / prueba ───────────────────────────────────────
# stratify=y: mantiene la misma proporción de churn en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Entrenar Random Forest ────────────────────────────────────────────────
# n_estimators=100: ensemble de 100 árboles de decisión
# n_jobs=-1: usa todos los núcleos del CPU
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# Verificar métricas
auc = roc_auc_score(y_test, modelo_rf.predict_proba(X_test)[:, 1])
print(f"✅ Modelo entrenado | AUC-ROC: {auc:.3f}")
print(f"   Entrenamiento: {X_train.shape[0]:,} clientes")
print(f"   Prueba:        {X_test.shape[0]:,} clientes")


---
## 2. ¿Qué hace SHAP internamente?

`TreeExplainer` es la versión de SHAP optimizada para modelos basados en árboles (Random Forest, XGBoost, etc.).

Para cada predicción, calcula **cuánto contribuyó cada variable** a alejarse de la probabilidad base:

```
P(churn del cliente X) = probabilidad_base + SHAP(var_1) + SHAP(var_2) + ... + SHAP(var_n)
```

- Un SHAP value **positivo** → la variable empuja hacia predecir churn
- Un SHAP value **negativo** → la variable empuja hacia predecir retención
- Un SHAP value **cercano a 0** → esa variable casi no afecta la predicción

In [ ]:
# ── Crear el explicador SHAP ──────────────────────────────────────────────
# TreeExplainer está optimizado para modelos basados en árboles (Random Forest,
# XGBoost, LightGBM, etc.). Es mucho más rápido que el KernelExplainer genérico.
explainer = shap.TreeExplainer(modelo_rf)

# ── Calcular los SHAP values ──────────────────────────────────────────────
# Calculamos los valores sobre el conjunto de PRUEBA (datos no vistos por el modelo).
# Esto puede tardar 1-2 minutos con ~2,000 clientes.
print("⏳ Calculando SHAP values... (puede tardar 1-2 minutos)")
shap_values = explainer.shap_values(X_test)

# Para clasificación binaria, shap_values es una lista de 2 arrays:
#   shap_values[0] → contribuciones hacia predecir clase 0 (se quedó)
#   shap_values[1] → contribuciones hacia predecir clase 1 (se fue) ← nos interesa esto
sv_churn = shap_values[1]         # Array de shape: (n_clientes, n_variables)
ev_churn = explainer.expected_value[1]   # Probabilidad base de churn

print(f"✅ SHAP values calculados")
print(f"   Dimensiones: {sv_churn.shape[0]:,} clientes × {sv_churn.shape[1]} variables")
print(f"   Probabilidad base de churn (sin info del cliente): {ev_churn:.1%}")


---
## 3. Análisis Global: ¿Qué variables más importan?

El análisis global responde: **¿en promedio, qué variables tienen mayor impacto en las predicciones?**

Dos visualizaciones complementarias:
- **Gráfico de barras**: magnitud del impacto (qué tan importante es cada variable)
- **Beeswarm plot**: dirección del impacto (si un valor alto o bajo de la variable sube o baja el riesgo)

In [ ]:
# ── Importancia Global de Variables (Barras) ──────────────────────────────
# Para cada variable, promedia el |SHAP value| de todos los clientes.
# Cuanto mayor la barra, más impacto tiene esa variable en las predicciones.

plt.figure(figsize=(10, 7))
shap.summary_plot(sv_churn, X_test, plot_type="bar", show=False)
plt.title(
    "Importancia Global — ¿Qué determina el riesgo de churn?",
    fontsize=13, pad=15
)
plt.xlabel("Impacto promedio en la predicción (|SHAP value| medio)")
plt.tight_layout()
plt.savefig("shap_importance_bar.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Dirección e Intensidad del Impacto (Beeswarm) ─────────────────────────
# Cada punto representa UN cliente en el set de prueba.
# Color: rojo = valor alto de esa variable | azul = valor bajo
# Posición X: derecha = empuja hacia churn | izquierda = empuja hacia retención

shap.summary_plot(sv_churn, X_test, show=False)
plt.title(
    "Dirección del Impacto — ¿Cómo empuja cada variable la predicción?",
    fontsize=13, pad=15
)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()


### 💡 Cómo leer el beeswarm plot

| Color | Significado |
|-------|-------------|
| 🔴 Rojo | Valor **alto** de esa variable para ese cliente |
| 🔵 Azul | Valor **bajo** de esa variable para ese cliente |
| → Derecha (+) | Empuja hacia **predecir churn** |
| → Izquierda (−) | Empuja hacia **predecir retención** |

**Ejemplo de lectura**: si los puntos rojos de `Total_Trans_Ct` están a la izquierda,
significa que muchas transacciones (valor alto) reduce el riesgo de churn.

---
## 4. Análisis Individual: ¿Por qué este cliente específico se va a ir?

El análisis local explica **una predicción concreta**.

El **waterfall plot** muestra paso a paso cómo cada variable mueve la probabilidad predicha:
- Parte inferior → probabilidad base (sin conocer nada del cliente)
- Barras rojas → variables que aumentan el riesgo de churn
- Barras azules → variables que lo reducen
- Parte superior → probabilidad final predicha para este cliente

In [ ]:
# ── Seleccionar el cliente con mayor probabilidad predicha de churn ────────
proba_churn = modelo_rf.predict_proba(X_test)[:, 1]
idx_max_riesgo = proba_churn.argmax()

prob_predicha = proba_churn[idx_max_riesgo]
etiqueta_real = "Churned ✓" if y_test.iloc[idx_max_riesgo] == 1 else "Retenido"

print(f"🎯 Cliente analizado: posición {idx_max_riesgo} en el set de prueba")
print(f"   Probabilidad predicha de churn: {prob_predicha:.1%}")
print(f"   Etiqueta real:                  {etiqueta_real}")
print()

# Top 3 características de este cliente
top_vars = ['Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Total_Trans_Amt']
print("Características clave de este cliente:")
print(X_test.iloc[idx_max_riesgo][top_vars].to_frame(name="Valor del cliente"))

# ── Waterfall Plot ─────────────────────────────────────────────────────────
# Muestra cómo cada variable mueve la predicción desde la probabilidad base
# hasta la predicción final para este cliente específico.
shap.waterfall_plot(
    shap.Explanation(
        values=sv_churn[idx_max_riesgo],         # SHAP values de este cliente
        base_values=ev_churn,                     # Probabilidad base
        data=X_test.iloc[idx_max_riesgo].values,  # Valores reales del cliente
        feature_names=list(X_test.columns)
    )
)


---
## 5. Análisis Profundo: ¿Cómo cambia el riesgo según el valor de cada variable?

Los **dependence plots** muestran la relación entre el valor de una variable y su impacto SHAP
para todos los clientes del set de prueba.

Esto responde: *¿A partir de qué umbral de transacciones el riesgo de churn empieza a dispararse?*

- Eje X → valor de la variable para ese cliente
- Eje Y → cuánto empuja esa variable hacia churn (+) o retención (−)
- La línea punteada gris en Y=0 es el punto neutro

In [ ]:
# ── Dependence Plots: relación valor-impacto para las 3 variables top ─────
# Cada punto = un cliente del set de prueba
# Eje X = valor de la variable para ese cliente
# Eje Y = SHAP value (cuánto empuja esa variable hacia churn o retención)
# Y = 0 (línea punteada) = la variable no afecta la predicción para ese cliente

variables_clave = ['Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Total_Trans_Amt']
titulos = [
    'Nº total de transacciones',
    'Cambio en transacciones (Q4/Q1)',
    'Monto total de transacciones'
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (var, titulo) in enumerate(zip(variables_clave, titulos)):
    shap.dependence_plot(
        var, sv_churn, X_test,
        ax=axes[i], show=False
    )
    axes[i].set_title(titulo, fontsize=11, fontweight='bold')
    axes[i].axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Neutro')

plt.suptitle(
    "¿Cómo cambia el riesgo de churn según el valor de cada variable?",
    fontsize=13, y=1.03
)
plt.tight_layout()
plt.savefig("shap_dependence.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Conclusiones y Recomendaciones de Negocio

### Hallazgos principales del análisis SHAP

| Variable clave | Insight | Acción recomendada |
|---------------|---------|-------------------|
| `Total_Trans_Ct` | Clientes con <40 transacciones en riesgo muy alto | Alertas tempranas + incentivos de uso |
| `Total_Ct_Chng_Q4_Q1` | Caída >30% en transacciones = señal crítica | Monitoreo mensual de tendencia |
| `Total_Revolving_Bal` | Saldo revolving alto reduce el riesgo | No incentivar pago total a clientes en riesgo |
| `Contacts_Count_12_mon` | Muchos contactos = señal de insatisfacción | Revisar calidad de atención al cliente |

### Validación del hallazgo principal del EDA

✅ El análisis SHAP **confirma cuantitativamente** lo que el EDA sugería cualitativamente:
el comportamiento transaccional (cómo usa el cliente la tarjeta) predice el churn
mucho mejor que las características demográficas (edad, género, educación).

### Próximos pasos sugeridos

1. **Dashboard de retención**: visualizar el riesgo y los drivers SHAP por cliente en tiempo real
2. **Scoring mensual**: recalcular el riesgo cada mes y alertar cuando suba significativamente
3. **Segmentación**: 3 niveles de riesgo (alto/medio/bajo) con campañas diferenciadas
4. **A/B testing**: piloto de retención basado en estos insights para medir ROI del modelo